In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

In [4]:
df = pd.read_csv('../data/sample_data.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df[['date_time']].rename(columns={'date_time':'datetime'}, inplace=True)
df['date'] = pd.to_datetime(df['date'])
print(f'数据加载完成，共{len(df)}条数据')

数据加载完成，共5025843条数据


In [6]:
analysis_date = df['date'].max() + pd.Timedelta(days = 1)
print(f"分析截止日期: {analysis_date.date()}")
buy_df = df[df['behavior_type'] == 'buy']
rfm = buy_df.groupby('user_id').agg(
    r = ('date',lambda x: (analysis_date - x.max()).days),
    f = ('behavior_type','count')
).reset_index()
user_activities = df.groupby('user_id').size().reset_index(name = 'm')
rfm = rfm.merge(user_activities, on = 'user_id',how = 'left')

print(f"有购买行为的用户数: {len(rfm):,}")
print("\nRFM值统计描述：")
print(rfm[['r', 'f', 'm']].describe().round(2))
print("\nR值分布（最近购买距今天数）：")
print(rfm['r'].value_counts().sort_index())
print("\nF值分布（购买次数）：")
print(rfm['f'].value_counts().sort_index())

分析截止日期: 2017-12-04
有购买行为的用户数: 34,021

RFM值统计描述：
              r         f         m
count  34021.00  34021.00  34021.00
mean       3.61      2.98    111.65
std        2.40      2.78     90.74
min        1.00      1.00      1.00
25%        2.00      1.00     46.00
50%        3.00      2.00     85.00
75%        5.00      4.00    150.00
max        9.00     57.00    763.00

R值分布（最近购买距今天数）：
r
1    8047
2    6815
3    4427
4    3726
5    3126
6    2473
7    2136
8    1851
9    1420
Name: count, dtype: int64

F值分布（购买次数）：
f
1     11598
2      7929
3      5102
4      3238
5      2091
6      1310
7       845
8       533
9       386
10      256
11      187
12      127
13       85
14       78
15       56
16       30
17       34
18       29
19       14
20       15
21       11
22       14
23        5
24        7
25        6
26        4
27        2
28        5
29        3
30        2
32        2
33        1
36        1
39        4
40        1
42        1
43        1
45        1
46        1
48        

In [10]:
rfm['R_score'] = pd.qcut(rfm['r'].rank(method='first'), q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(
    rfm['f'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]
).astype(int)
rfm['M_score'] = pd.qcut(rfm['m'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print("RFM打分结果统计：")
print("\nR_score 分布：")
print(rfm['R_score'].value_counts().sort_index())
print("\nF_score 分布：")
print(rfm['F_score'].value_counts().sort_index())
print("\nM_score 分布：")
print(rfm['M_score'].value_counts().sort_index())
print("\nRFM总分分布：")
print(rfm['RFM_total'].value_counts().sort_index())

print("\nRFM各维度平均分：")
print(f"  R_score 平均: {rfm['R_score'].mean():.2f}")
print(f"  F_score 平均: {rfm['F_score'].mean():.2f}")
print(f"  M_score 平均: {rfm['M_score'].mean():.2f}")
print(f"  总分平均: {rfm['RFM_total'].mean():.2f}")

RFM打分结果统计：

R_score 分布：
R_score
1    6804
2    6804
3    6804
4    6804
5    6805
Name: count, dtype: int64

F_score 分布：
F_score
1    6805
2    6804
3    6804
4    6804
5    6804
Name: count, dtype: int64

M_score 分布：
M_score
1    6851
2    6775
3    6868
4    6777
5    6750
Name: count, dtype: int64

RFM总分分布：
RFM_total
3      559
4     1454
5     2130
6     3078
7     3914
8     4040
9     3979
10    4080
11    3586
12    2868
13    2204
14    1400
15     729
Name: count, dtype: int64

RFM各维度平均分：
  R_score 平均: 3.00
  F_score 平均: 3.00
  M_score 平均: 2.99
  总分平均: 8.99


In [11]:
r_avg = rfm['R_score'].mean()
f_avg = rfm['F_score'].mean()
m_avg = rfm['M_score'].mean()

print(f"R_score 均值: {r_avg:.2f}  (高于均值=R高，低于=R低)")
print(f"F_score 均值: {f_avg:.2f}  (高于均值=F高，低于=F低)")
print(f"M_score 均值: {m_avg:.2f}  (高于均值=M高，低于=M低)")
def segment_user(row):
    """根据RFM得分进行用户分层"""
    r_high = row['R_score'] >= r_avg
    f_high = row['F_score'] >= f_avg
    m_high = row['M_score'] >= m_avg
    if r_high and f_high and m_high:
        return '重要价值用户'
    elif r_high and f_high and not m_high:
        return '重要发展用户'
    elif r_high and not f_high and m_high:
        return '重要保持用户'
    elif r_high and not f_high and not m_high:
        return '新用户'
    elif not r_high and f_high and m_high:
        return '重要召回用户'
    elif not r_high and f_high and not m_high:
        return '一般保持用户'
    elif not r_high and not f_high and m_high:
        return '潜在用户'
    else:
        return '流失用户'
rfm['user_segment'] = rfm.apply(segment_user, axis=1)
segment_stats = rfm['user_segment'].value_counts().reset_index()
segment_stats.columns = ['用户层级', '用户数']
segment_stats['占比(%)'] = (
    segment_stats['用户数'] / segment_stats['用户数'].sum() * 100
).round(2)

print("\n" + "=" * 50)
print("用户分层结果")
print("=" * 50)
print(segment_stats.to_string(index=False))

print(f"\n总用户数: {len(rfm):,}")

R_score 均值: 3.00  (高于均值=R高，低于=R低)
F_score 均值: 3.00  (高于均值=F高，低于=F低)
M_score 均值: 2.99  (高于均值=M高，低于=M低)

用户分层结果
  用户层级  用户数  占比(%)
重要召回用户 7276  21.39
重要价值用户 6365  18.71
  潜在用户 5113  15.03
  流失用户 4764  14.00
重要发展用户 3512  10.32
一般保持用户 3259   9.58
   新用户 2091   6.15
重要保持用户 1641   4.82

总用户数: 34,021
